# 03 - Local Embeddings and ChromaDB Indexing

Generates dense vector embeddings using `all-MiniLM-L6-v2` and persists complete resume documents into ChromaDB.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_data
from src.resume_parser import parse_all_resumes
from src.embeddings import get_embedding_model, embed_query
from src.vector_store import build_index, load_collection, get_collection_stats

## 1. Load Local Embedding Model

In [ ]:
embedder = get_embedding_model()
test_vec = embed_query("Data Analyst with SQL")
print(f"Embedding dimension: {len(test_vec)}")
print(f"First 5 vector components: {test_vec[:5]}")

## 2. Build or Load ChromaDB Index

In [ ]:
raw_records = load_raw_data()
profiles = parse_all_resumes(raw_records)

# Build or reuse persistent index
doc_count = build_index(profiles, rebuild=False)
stats = get_collection_stats()
print(f"Collection status: {stats}")

## 3. Verify Document Retrieval

In [ ]:
collection = load_collection()
sample_query_res = collection.query(
    query_embeddings=[test_vec],
    n_results=2,
    include=["metadatas", "distances"]
)
print("Top match metadata:")
print(sample_query_res["metadatas"][0][0])
print(f"Cosine distance: {sample_query_res['distances'][0][0]:.4f}")